# Lecture: Tiny GPT — Predicting the Next Character (Tokenizer → GPT → Sampling)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain next-token prediction: what the input, target, and loss are (§4, §7).
- Explain what the tokenizer, embeddings, block, and LM head each do (§4–§7).
- Explain perplexity and temperature sampling in plain words (§7–§8).


## The big idea

Think of phone autocomplete, but for every character: the model reads "the cat sat on the m" and must bet on the next letter. Training turns text into number lists (**tokenizer**: 'a' → 5, 'b' → 12… — the numbers are **token ids** drawn from the **vocabulary**), looks up a learned fingerprint per number (**token embedding**) plus a position fingerprint (**positional embedding**, so order matters within the **context window**), and mixes them through stacked **blocks** (each = norm → causal attention → residual → norm → MLP → residual). The final state becomes one score per vocabulary item (**LM head** → **logits**), **cross-entropy loss** punishes probability placed on wrong letters, and **perplexity** — exp(loss) — reports "how many letters wide the confusion is". To write new text, the model samples its own distribution (**temperature** cools it toward safe picks or heats it toward wild ones) and feeds each pick back in.


In [ ]:
import numpy as np  # bring in numpy for ids, softmax, and sampling
import matplotlib.pyplot as plt  # bring in pyplot for the distribution bars
import random  # bring in random for stdlib seed parity
np.random.seed(11)  # fix numpy seed so sampling is identical every run
random.seed(11)  # fix stdlib seed for full reproducibility
text = 'the cat'  # tiny toy string to tokenize by hand
vocab = sorted(set(text))  # build the toy vocab: unique chars, sorted
stoi = {ch: i for i, ch in enumerate(vocab)}  # char -> id lookup table
itos = {i: ch for ch, i in stoi.items()}  # id -> char reverse table
ids = [stoi[ch] for ch in text]  # tokenize: map each char to its id
print('vocab:', vocab)  # the 6-character menu: space, a, c, e, h, t
print('text -> ids:', ids)  # token ids for 'the cat'
logits = np.array([2.0, 0.5, 0.2, 1.0, 0.1, 0.3])  # toy LM-head scores, one per vocab char
e = np.exp(logits - logits.max())  # stable exponentiate for softmax
probs = e / e.sum()  # next-char probability distribution: sums to 1
print('uniform baseline loss ln(V) =', round(float(np.log(len(vocab))), 4))  # guessing loss with 6 chars
print('top next-char prob:', round(float(probs.max()), 3))  # best guess share of the distribution
fig, ax = plt.subplots(figsize=(7, 3))  # make one bar panel
ax.bar([repr(itos[i]) for i in range(len(vocab))], probs, color='steelblue')  # one bar per char
ax.set_ylim(0, 1)  # fix the probability range
ax.set_title("Next-char bets after 'the ca' (must sum to 1)")  # panel title
ax.set_ylabel('probability')  # label the y axis
fig.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 numbers the text, §5–§6 build the model, §7–§8 score and sample it.

| Term | One-liner | Section |
|---|---|---|
| Tokenizer / vocabulary | Turns text into id lists; vocabulary is the full char menu (lab: 65 chars) | §4 |
| Token ids | The numbers standing in for characters that the model actually reads | §4 |
| Token embedding | Learned fingerprint vector per id (what the character means) | §5 |
| Positional embedding | Learned fingerprint per position (order signal; dropping it costs +0.12 loss) | §5 |
| Context window | How many past tokens the model may look at (lab: 48) | §5 |
| Block (LN + MHA + MLP) | One repeatable mixer unit; the lab stacks 4 (width 96, 4 heads) | §6 |
| LM head / logits | Final layer turning hidden state into one score per vocabulary item | §7 |
| Cross-entropy loss | Punishment for probability on wrong next tokens (lower = better) | §7 |
| Perplexity | exp(loss): effective confused-between options (lab: 11.4 vs 61) | §7 |
| Temperature sampling | Softens (hot) or sharpens (cool) the distribution before picking | §8 |


### §4 Tokenizer and token ids — text becomes numbers

**In one sentence: the tokenizer maps every character to an integer id from the vocabulary, so the model never sees letters — only id lists it can look up and do math on.**

Our toy vocab is the 6 sorted characters of "the cat" (space, a, c, e, h, t), giving ids `[5, 4, 3, 0, 2, 1, 5]` — note the two t's share id 5 (same char, same id, always). Training pairs come from one shift: input x = all ids but the last, target y = all ids but the first, so the model always bets each next char from the past only. Watch it: the x/y split plus a decode round-trip proving `decode(encode(s)) == s` — tokenization loses nothing.


In [ ]:
import numpy as np  # reuse numpy for the id arrays
np.random.seed(11)  # fix the seed for reproducibility
text = 'the cat'  # same toy string as the anchor
vocab = sorted(set(text))  # 6-char menu: [' ', 'a', 'c', 'e', 'h', 't']
stoi = {ch: i for i, ch in enumerate(vocab)}  # char -> id map
itos = {i: ch for ch, i in stoi.items()}  # id -> char map
ids = [stoi[ch] for ch in text]  # tokenize the string
x = np.array(ids[:-1])  # model input: all ids except the last
y = np.array(ids[1:])  # target: input shifted one step ahead
print('text -> ids:', ids)  # expect [5, 4, 3, 0, 2, 1, 5]: both t's are 5
print('input  x:', list(x))  # expect [5, 4, 3, 0, 2, 1]: 'the ca'
print('target y:', list(y))  # expect [4, 3, 0, 2, 1, 5]: 'he cat'
back = ''.join(itos[i] for i in ids)  # decode back to text
print('round-trip:', repr(back), '== original:', back == text)  # must be True: nothing lost


### §5 Embeddings and positions — numbers become vectors

**In one sentence: the token embedding looks up a learned fingerprint vector per id, the positional embedding looks up a learned vector per position, and their sum (inside a 48-token context window) is what the blocks actually read.**

Two tables, one addition: with vocab 6 and width 4, id 5 at position 2 becomes `tok[5] + pos[2]` — meaning plus order, fused before any mixing happens. Order has to be injected because attention itself is order-blind (§6's blocks would treat "cat" and "act" identically without it) — the lab proves the point by deleting positions and watching validation loss rise by +0.12. Watch it: the lookup diagram plus the printed `tok[5] + pos[2]` sum.


In [ ]:
import numpy as np  # reuse numpy for the embedding tables
import matplotlib.pyplot as plt  # reuse pyplot for the lookup diagram
np.random.seed(11)  # fix the seed so the tables are identical every run
V, d, T = 6, 4, 8  # toy sizes: 6 chars, width 4, window 8 (lab: 65, 96, 48)
tok = np.round(np.random.normal(size=(V, d)), 2)  # token table: one fingerprint row per id
pos = np.round(np.random.normal(size=(T, d)), 2)  # position table: one fingerprint row per slot
e = tok[5] + pos[2]  # id 5 ('t') at slot 2: meaning + order
print('tok table', tok.shape, '+ pos table', pos.shape, '-> vector', e.shape)  # (6,4) + (8,4) -> (4,)
print('tok[5]      :', tok[5])  # the meaning fingerprint
print('pos[2]      :', pos[2])  # the order fingerprint
print('input vector:', np.round(e, 2))  # the fused vector the blocks read
fig, ax = plt.subplots(figsize=(8, 3))  # make one panel for the lookup diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 3)  # fix placement range
ax.text(1.5, 1.5, 'tok table\n6 x 4\n(meaning)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))  # meaning box
ax.text(4.2, 1.5, 'pos table\n8 x 4\n(order)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # order box
ax.text(7.5, 1.5, 'e = tok[id]\n+ pos[t]\n(width 4)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # fusion box
ax.annotate('', xy=(6.3, 1.5), xytext=(5.4, 1.5), arrowprops=dict(arrowstyle='->'))  # tables into fusion
ax.annotate('', xy=(2.9, 1.5), xytext=(2.4, 1.5), arrowprops=dict(arrowstyle='->'))  # meaning into fusion path
ax.set_title('Embeddings: meaning + order, added before any mixing')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §6 The block — LN, attention, MLP, residuals

**In one sentence: a block normalizes, mixes tokens with causal attention, adds the result back (residual 1), normalizes again, transforms each position with an MLP, and adds back again (residual 2) — and the lab stacks 4 such blocks.**

Each half follows one pattern: `x = x + sublayer(norm(x))`. LayerNorm steadies the inputs (attention is happiest near unit scale), the sublayer does the work, and the `+ x` residual is a gradient highway — even if a sublayer outputs garbage early in training, the original signal sails through untouched, so deep stacks stay trainable. The MLP widens 4× inside (96 → 384 → 96 in the lab) to give each position private compute after attention's mixing. Watch it: the box diagram plus a toy residual `x + sub(x) = [1.5, 1.0]`.


In [ ]:
import numpy as np  # reuse numpy for the residual arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the block diagram
np.random.seed(11)  # fix the seed for reproducibility
x = np.array([1.0, 2.0])  # toy hidden state entering a sublayer
sub = np.array([0.5, -1.0])  # toy sublayer output (attention or MLP branch)
out = x + sub  # THE residual: add the branch back to the highway
print('x       :', x)  # the untouched highway signal
print('sublayer:', sub)  # the branch's contribution (can be anything, even zeros early on)
print('x + sub :', out)  # expect [1.5, 1.0]: highway preserved, branch added
fig, ax = plt.subplots(figsize=(10, 3))  # make one wide panel for the block
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix placement range
ax.set_ylim(0, 3)  # fix placement range
ax.text(1.2, 1.5, 'in x', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # entry box
ax.text(3.6, 1.5, 'LN', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))  # norm box
ax.text(6.0, 1.5, 'causal MHA\n(mix tokens)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))  # mixer box
ax.text(8.4, 1.5, '+residual', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # highway box
ax.text(10.8, 1.5, 'LN+MLP\n+residual', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # second half box
for xa, xb in [(1.8, 2.9), (4.2, 5.0), (7.0, 7.6), (9.2, 9.9)]:  # gaps between boxes
    ax.annotate('', xy=(xb, 1.5), xytext=(xa, 1.5), arrowprops=dict(arrowstyle='->'))  # forward arrow
ax.plot([1.2, 8.4], [0.6, 0.6], color='red', linewidth=1.5)  # red highway line under the first half
ax.text(4.8, 0.3, 'red = gradient highway: x sails through untouched', fontsize=8, color='darkred')  # highway note
ax.set_title('One block (x4 in the lab): norm -> mix -> add -> norm -> MLP -> add')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §7 LM head, cross-entropy, perplexity — betting and grading

**In one sentence: the LM head turns the final hidden state into one logit per vocabulary char, cross-entropy grades the bet with `−log(probability of the true next char)`, and perplexity = exp(loss) translates the grade into "confused between this many options".**

The uniform baseline anchors everything: guessing blindly over V chars gives loss ln(V) (lab: ln(65) ≈ 4.17; our toy: ln(4) ≈ 1.39). Watch it: toy scores for a/b/c/d make the model bet ~0.73 on 'a' — if 'a' is truly next, loss is −log(0.73) ≈ 0.31 and perplexity ≈ 1.37 (choosing among ~1.4 options, far sharper than the uniform 4.0). The lab's trained 0.46M-param net reaches the same idea at scale: val loss ≈ 2.43, perplexity ≈ 11.4 against a uniform 61.


In [ ]:
import numpy as np  # reuse numpy for softmax, loss, and perplexity
import matplotlib.pyplot as plt  # reuse pyplot for the next-char bars
import math  # reuse math for log/exp in the grading
np.random.seed(11)  # fix the seed for reproducibility
logits = np.array([3.0, 1.5, 0.5, 0.2])  # toy LM-head scores for chars a/b/c/d
labels = ['a', 'b', 'c', 'd']  # char labels for the bars
e = np.exp(logits - logits.max())  # stable exponentiate
p = e / e.sum()  # next-char distribution: sums to 1
true = 0  # suppose the true next char is 'a'
ce = -math.log(p[true])  # cross-entropy: -log(probability of truth)
ppl = math.exp(ce)  # perplexity: effective confused-between options
print('probs:', np.round(p, 3))  # the model's bets: 'a' leads
print('loss -log(p_true):', round(ce, 3))  # expect ~0.31: confident-and-right is cheap
print('perplexity:', round(ppl, 2), '(uniform would be 4.0)')  # expect ~1.37 vs blind 4.0
print('uniform baseline ln(4):', round(math.log(4), 3))  # expect 1.386: the score to beat
fig, ax = plt.subplots(figsize=(6, 3.5))  # make one panel for the bets
ax.bar(labels, p, color='steelblue')  # one bar per candidate char
ax.set_ylim(0, 1)  # fix the probability range
ax.set_title("Next-char bets: 'a' leads at ~0.73")  # title carries the bet
ax.set_ylabel('probability')  # label the y axis
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §8 Temperature — cool heads vs hot takes

**In one sentence: temperature divides the logits before sampling — T < 1 sharpens the distribution (safe, verse-like picks), T > 1 flattens it (wilder, surprising picks) — without changing the ranking.**

Same scores, different moods: our a/b/c/d bets go from a peaked cool distribution (top char ~0.86 at T = 0.7) to a flatter hot one (top char ~0.66 at T = 1.2). Greedy decoding is the T → 0 limit (always the argmax); real sampling rolls the dice, so the lab saves both styles into `results/samples.txt` — cool reads like chanting verse, hot like inventive babble. Watch it: the overlaid bars show probability mass draining from the winner to the underdogs as T rises.


In [ ]:
import numpy as np  # reuse numpy for softmax, sampling, and the curve
import matplotlib.pyplot as plt  # reuse pyplot for the temperature + loss panels
np.random.seed(11)  # fix the seed so samples and curve are identical every run
logits = np.array([3.0, 1.5, 0.5, 0.2])  # same toy scores as §7
labels = ['a', 'b', 'c', 'd']  # char labels
def sample_temp(logits, T):  # temperature-scaled distribution
    s = logits / T  # divide by T: hot flattens, cool sharpens
    e = np.exp(s - s.max())  # stable exponentiate of scaled scores
    p = e / e.sum()  # normalize to probabilities
    return p  # return the sampling distribution
p_cool = sample_temp(logits, 0.7)  # cool distribution: confident, verse-like
p_hot = sample_temp(logits, 1.2)  # hot distribution: wilder, surprising
print('T=0.7 probs:', np.round(p_cool, 3))  # peaked: winner takes most
print('T=1.2 probs:', np.round(p_hot, 3))  # flatter: underdogs gain share
print('greedy (T->0) always picks:', labels[int(np.argmax(logits))])  # argmax: 'a', no dice involved
iters = np.arange(0, 301, 50)  # x axis: training iterations 0..300
loss = 4.28 - 1.9 * (1 - np.exp(-iters / 120.0))  # sketch of decay 4.28 -> ~2.4 (lab shape)
print('sketch init loss:', round(float(loss[0]), 2), '(uniform ln61 = 4.11: starts near guessing)')  # start
print('sketch final loss:', round(float(loss[-1]), 2))  # end of the sketch curve
fig, ax = plt.subplots(1, 2, figsize=(9, 3))  # two panels: temperature + training sketch
ax[0].bar(labels, p_cool, alpha=0.7, label='T=0.7 cool')  # cool bars
ax[0].bar(labels, p_hot, alpha=0.5, label='T=1.2 hot')  # hot bars overlaid
ax[0].set_ylim(0, 1)  # fix the probability range
ax[0].set_title('temperature reshapes the bets')  # title the left panel
ax[0].legend()  # show which bars are cool vs hot
ax[1].plot(iters, loss, marker='o')  # sketch training curve
ax[1].set_xlabel('iters')  # label the x axis
ax[1].set_ylabel('val loss')  # label the y axis
ax[1].set_title('300 iters: 4.28 -> ~2.5 (lab shape)')  # title the right panel
fig.suptitle('Cooler = safer text; training = shrinking confusion')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


## How it all fits together — the system map (§9)

One generated character travels the whole line: raw **text** is **tokenized** to ids (§4), ids become meaning+order **vectors** (§5), vectors ride 4 **blocks** of mix-and-add (§6), the final state becomes **logits** over the vocab (§7), **temperature** reshapes them (§8), one char is sampled — and that char rejoins the input for the next lap. Training is the same line with the loop cut: inputs bet every next char at once (thanks to §6's causal mask) and cross-entropy grades all bets in parallel. The trace below plays one lap: the §7 distribution's greedy pick is 'a', which would be appended to the context for the next prediction.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(11)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('text', 0.8), ('ids §4\ntokenize', 3), ('vectors §5\nemb+pos', 5.4), ('blocks §6\nx4 mix', 7.8), ('logits §7\n-> sample §8', 10.6)]  # pipeline stations
colors = ['wheat', 'lightblue', 'lightgreen', 'plum', 'steelblue']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(5.8, 3.5, 'GENERATE: text -> ids -> vectors -> mix -> bets -> pick -> loop back', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(1.4, 2.2), (3.9, 4.5), (6.3, 6.9), (8.7, 9.6)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.annotate('', xy=(3.0, 1.1), xytext=(10.6, 1.1), arrowprops=dict(arrowstyle='->', color='red', lw=2))  # red loop-back arrow
ax.text(6.8, 0.6, 'sample loop: append the pick, predict again (§8)', ha='center', fontsize=9, color='darkred')  # loop note
ax.set_title('System map: one generated char through every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the one-distribution trace
np.random.seed(11)  # fix the seed (arrays below are hardcoded; habit kept)
logits = np.array([3.0, 1.5, 0.5, 0.2])  # one LM-head output for a/b/c/d (§7)
labels = ['a', 'b', 'c', 'd']  # candidate chars
e = np.exp(logits - logits.max())  # stable exponentiate
p = e / e.sum()  # sampling distribution (§7)
greedy = labels[int(np.argmax(p))]  # T->0 pick: the argmax (§8)
pick = labels[int(np.argmax(np.random.multinomial(1, p)))]  # one T=1.0 dice roll
print('distribution:', np.round(p, 3))  # the bets (§7)
print('greedy pick:', repr(greedy))  # expect 'a': no dice involved
print('sampled pick (T=1.0, seeded):', repr(pick))  # one honest dice roll
print('loop: append the pick to the ids and predict again (§9)')  # the autoregressive step


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, on TinyShakespeare:

1. **Cell 1 (setup)** — seeds, device pick, `os.makedirs('./results')`. Same seed habit as this lecture.
2. **Cell 2 (data)** — downloads TinyShakespeare (120k chars, vocab 65, uniform baseline ln(65) ≈ 4.17) with a synthetic fallback — tokenized exactly like §4 into ~108k train / ~12k val ids, batched as shifted (x, y) pairs of 48-token windows.
3. **Cell 3 (model)** — `TinyGPT` = token + position embeddings (§5), 4 pre-norm blocks (§6: causal MHA + widen-4 MLP with residuals), final LN + LM head (§7) — ~0.46M params, starting near the uniform baseline.
4. **Cell 4 (training + sampling)** — AdamW (3e-4, 300 iters × 1152 tokens) drives train ≈ 2.37 / val ≈ 2.43 (perplexity ≈ 11.4 vs 61, §7); greedy/temperature sampling feeds its own output through the 48-token window into `results/samples.txt` (§8–§9).
5. **Cell 5 (experiments)** — no-position models lose +0.12 (bag-of-chars can't do order, §5), and the context probe finds a 16-token sweet spot — extra context hurts a 0.5M model, previewing P12's capacity-vs-context math.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **In next-token training, what are the input and the target for the text "hello"?**

*Answer: input = ids of "hell", target = ids of "ello" (input shifted one step ahead, §4) — the model bets each next char from the past only, and `decode(encode(s)) == s` proves the numbering loses nothing.*

2. **Perplexity 11.4 on a 65-char vocab — is that good, and what is the baseline?**

*Answer: much better than guessing (§7) — uniform guessing gives loss ln(65) ≈ 4.17, i.e. perplexity 65, so 11.4 means confusion narrowed to ~11 effective options per character. Our toy shows the same idea: loss 0.31 → perplexity 1.37 vs blind 4.0.*

3. **What changes when sampling temperature goes from 0.7 to 1.2?**

*Answer: the distribution flattens without re-ranking (§8) — our winner's share drains from ~0.86 to ~0.66, so rare letters get picked more often and output turns wilder (exactly the lab's `samples.txt` comparison).*
